# 00 · Organização e Configuração do Ambiente — CineData Analytics

Este notebook é responsável pelo provisionamento e verificação do ambiente local de dados:
1. Criação das estruturas de diretórios das camadas do *data lake* (`landing`, `bronze`, `silver`, `gold`).
2. Validação pré-voo dos artefatos brutos na camada *Landing*.
3. Checagens de integridade e inventário de arquivos de entrada para o pipeline.

Cada notebook subsequente do pipeline (`01_Landing_to_Bronze`, `02_Bronze_to_Silver`, `03_Silver_to_Gold`)
é integralmente auto-contido em suas importações, sessão Spark, constantes de negócio e funções de transformação.

In [1]:
from pathlib import Path

# Resolução do diretório base do repositório
WORKSPACE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = WORKSPACE_DIR / "data"
LANDING_DIR = DATA_DIR / "landing"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Criação estruturada de todas as camadas do lakehouse
for target_directory in [LANDING_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    target_directory.mkdir(parents=True, exist_ok=True)
    print(f"Diretório operacional confirmado: {target_directory}")


Diretório operacional confirmado: /home/miguelsb/workspace/visagio/data/landing
Diretório operacional confirmado: /home/miguelsb/workspace/visagio/data/bronze
Diretório operacional confirmado: /home/miguelsb/workspace/visagio/data/silver
Diretório operacional confirmado: /home/miguelsb/workspace/visagio/data/gold


In [2]:
EXPECTED_LANDING_FILES = [
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv",
]

# Checagem de presença física e volumetria dos arquivos da Landing Zone
existing_files = {
    file_entry.name: file_entry.stat().st_size
    for file_entry in LANDING_DIR.glob("*.csv")
}
missing_files = [filename for filename in EXPECTED_LANDING_FILES if filename not in existing_files]

if missing_files:
    print(f"[ALERTA] Arquivos ausentes na Landing Zone ({LANDING_DIR}):")
    for missing_filename in missing_files:
        print(f"  - ❌ {missing_filename}")
else:
    print(f"[SUCESSO] Todos os {len(EXPECTED_LANDING_FILES)} arquivos obrigatórios estão disponíveis:")
    for filename in EXPECTED_LANDING_FILES:
        file_size_kb = existing_files[filename] / 1024
        print(f"  - ✅ {filename} ({file_size_kb:.1f} KB)")


[SUCESSO] Todos os 5 arquivos obrigatórios estão disponíveis:
  - ✅ movies_info_TMDB_IMDB.csv (33129.0 KB)
  - ✅ movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - ✅ movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - ✅ credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - ✅ movies_reviews.csv (1878.4 KB)
